# Drone Detection & Tracking — YOLO11 + ByteTrack

Runtime: T4 GPU.

In [ ]:
%cd /content
!rm -rf drone-detection-tracking
!git clone -q https://github.com/semanur-s/drone-detection-tracking.git
%cd /content/drone-detection-tracking
!pip install -q -r requirements.txt

import os, random, shutil, json
from pathlib import Path
import cv2, matplotlib.pyplot as plt
os.makedirs('assets', exist_ok=True)
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

## 1. Data

In [ ]:
!python src/prepare_data.py --train-batches 1 --test-batches 1 --max-train 8000 --max-test 2000 --out data

In [ ]:
imgs = sorted(p for p in Path('data/drone/train/images').iterdir() if p.suffix.lower() in IMG_EXT)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.ravel(), random.Random(0).sample(imgs, 8)):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); h, w = im.shape[:2]
    lbl = p.parent.parent / 'labels' / (p.stem + '.txt')
    for line in (lbl.read_text().splitlines() if lbl.exists() else []):
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        cv2.rectangle(im, (x1, y1), (int(x1 + bw*w), int(y1 + bh*h)), (255, 0, 0), 2)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.savefig('assets/ornek_veriler.png', dpi=100); plt.show()

## 2. Training

In [ ]:
!python src/train.py --data data/drone.yaml --model yolo11n.pt --epochs 50 --imgsz 640 --batch 16 --device 0
WEIGHTS = open('results/best_weights.txt').read().strip()
run_dir = Path(WEIGHTS).parent.parent
shutil.copy(run_dir / 'results.png', 'assets/egitim_egrileri.png')
from IPython.display import Image, display
display(Image('assets/egitim_egrileri.png', width=1000))

## 3. Evaluation

In [ ]:
!python src/evaluate.py --weights $WEIGHTS --data data/drone.yaml

In [ ]:
from ultralytics import YOLO
model = YOLO(WEIGHTS)
test_imgs = sorted(p for p in Path('data/drone/test/images').iterdir() if p.suffix.lower() in IMG_EXT)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.ravel(), random.Random(1).sample(test_imgs, 8)):
    r = model.predict(str(p), conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis('off')
plt.tight_layout(); plt.savefig('assets/test_tahminleri.png', dpi=100); plt.show()
for src, dst in [('confusion_matrix.png', 'confusion_matrix.png'), ('BoxPR_curve.png', 'pr_egrisi.png')]:
    p = Path('runs/detect/test_eval') / src
    if p.exists(): shutil.copy(p, f'assets/{dst}')

## 4. Tracking
Upload a video showing a drone in flight.

In [ ]:
from google.colab import files
up = files.upload()
VIDEO = list(up.keys())[0]

In [ ]:
!python src/track.py --weights $WEIGHTS --source "$VIDEO" --tracker configs/bytetrack_drone.yaml
cap = cv2.VideoCapture('results/tracked.mp4'); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, n // 2); ok, frame = cap.read(); cap.release()
cv2.imwrite('assets/takip_karesi.jpg', frame)
plt.figure(figsize=(12, 7)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## 5. ONNX export & CPU benchmark

In [ ]:
!python src/export_benchmark.py --weights $WEIGHTS --images data/drone/test/images --n 100

## 6. Results

In [ ]:
for f in ['test_metrics.json', 'track_stats.json', 'benchmark.json']:
    p = Path('results') / f
    if p.exists(): print(f'--- {f}'); print(p.read_text())
!zip -qr sonuclar.zip results assets
files.download('sonuclar.zip')